# GCS full-kitchen pick, navigate, and place

In [1]:
import math
import threading
import time
from functools import partial

import plotly.graph_objects as go
import rclpy

from coraplex.datastructures.dataclasses import Context
from coraplex.datastructures.enums import Arms, ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.execution_environment import simulated_robot
from coraplex.locations.factories import reachability_location
from coraplex.plans.factories import code, sequential
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from coraplex.robot_plans.actions.core.placing import PlaceAction
from coraplex.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from coraplex.testing import setup_world
from coraplex.view_manager import ViewManager
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.datastructures.definitions import TorsoState
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Milk, Table
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix, Point3, Pose
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.geometry import BoundingBox
from semantic_digital_twin.world_description.graph_of_convex_sets import GraphOfConvexSets
from semantic_digital_twin.world_description.shape_collection import BoundingBoxCollection
from semantic_digital_twin.world_description.world_entity import Body

In [2]:
world = setup_world()
robot = PR2.from_world(world)
initial_base_pose = robot.root.global_pose

milk = world.get_body_by_name("milk.stl")
cereal = world.get_body_by_name("breakfast_cereal.stl")
milk_annotation = world.get_semantic_annotations_by_type(Milk)[0]
main_table = Table(root=world.get_body_by_name("table_area_main"))

with world.modify_world():
    world.add_semantic_annotation(main_table)
    supporting_surface = main_table.calculate_supporting_surface()

assert supporting_surface is not None

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [3]:
if not rclpy.ok():
    rclpy.init()

ros_node = rclpy.create_node("coraplex_gcs_full_kitchen")
ros_thread = threading.Thread(target=rclpy.spin, args=(ros_node,), daemon=True)
ros_thread.start()

In [4]:
tf_publisher = TFPublisher(_world=world, node=ros_node)
visualization_publisher = VizMarkerPublisher(_world=world, node=ros_node)

print(f"RViz fixed frame: {world.root.name}")
print(f"MarkerArray topic: {visualization_publisher.topic_name}")

RViz fixed frame: apartment/apartment_root
MarkerArray topic: /semworld/viz_marker


In [5]:
context = Context(world=world, robot=robot, ros_node=ros_node)
assert context.evaluate_conditions

grasp_description = GraspDescription(
    approach_direction=ApproachDirection.FRONT,
    vertical_alignment=VerticalAlignment.NoAlignment,
    end_effector=ViewManager.get_end_effector_view(Arms.RIGHT, robot),
)
preparation = sequential(
    [ParkArmsAction(Arms.BOTH), MoveTorsoAction(TorsoState.HIGH)],
    context=context,
)
with simulated_robot:
    preparation.perform()

In [6]:
def navigation_point(base_pose: Pose, height: float, reference_frame: Body) -> Point3:
    """Project a robot base pose onto the GCS navigation layer."""
    return Point3(
        x=base_pose.x,
        y=base_pose.y,
        z=height,
        reference_frame=reference_frame,
    )


def placement_pose_from_surface_sample(
    sampled_point: Point3,
    table: Table,
    object_body: Body,
    world: World,
) -> Pose:
    """Create an object pose from a sampled free table-surface point."""
    sampled_world_point = world.transform(sampled_point, world.root)
    table_top_height = float(table.root.global_pose.z + table.min_max_points[1].z)
    object_half_height = float(object_body.combined_mesh.extents[2] / 2)
    surface_orientation = world.transform(
        Pose(reference_frame=table.supporting_surface),
        world.root,
    ).to_quaternion()
    return Pose.from_xyz_quaternion(
        sampled_world_point.x,
        sampled_world_point.y,
        table_top_height + object_half_height,
        *surface_orientation.to_list(),
        reference_frame=world.root,
    )


def filter_path(
    raw_path: list[Point3],
    minimum_waypoint_distance: float,
) -> list[Point3]:
    """Remove consecutive decomposition waypoints below a distance threshold."""
    filtered_path = [raw_path[0]]
    for candidate in raw_path[1:-1]:
        previous = filtered_path[-1]
        candidate_distance = math.hypot(
            float(candidate.x - previous.x),
            float(candidate.y - previous.y),
        )
        if candidate_distance >= minimum_waypoint_distance:
            filtered_path.append(candidate)
    filtered_path.append(raw_path[-1])
    return filtered_path


def interpolate_path(
    path: list[Point3],
    step_distance: float,
    final_yaw: float,
    reference_frame: Body,
) -> list[Pose]:
    """Convert a GCS path into closely spaced base poses for RViz playback."""
    poses = []
    for segment_start, segment_end in zip(path, path[1:]):
        start_x = float(segment_start.x)
        start_y = float(segment_start.y)
        distance_x = float(segment_end.x) - start_x
        distance_y = float(segment_end.y) - start_y
        segment_distance = math.hypot(distance_x, distance_y)
        segment_step_count = max(1, math.ceil(segment_distance / step_distance))
        segment_yaw = math.atan2(distance_y, distance_x)

        for segment_step_index in range(1, segment_step_count + 1):
            progress = segment_step_index / segment_step_count
            poses.append(
                Pose.from_xyz_rpy(
                    x=start_x + distance_x * progress,
                    y=start_y + distance_y * progress,
                    z=0.0,
                    yaw=segment_yaw,
                    reference_frame=reference_frame,
                )
            )

    final_point = path[-1]
    poses[-1] = Pose.from_xyz_rpy(
        x=final_point.x,
        y=final_point.y,
        z=0.0,
        yaw=final_yaw,
        reference_frame=reference_frame,
    )
    return poses

In [7]:
navigation_height = 0.2
search_space = BoundingBoxCollection(
    [
        BoundingBox(
            min_x=0.2,
            max_x=6.5,
            min_y=0.2,
            max_y=5.8,
            min_z=0.15,
            max_z=0.25,
            origin=HomogeneousTransformationMatrix(reference_frame=world.root),
        )
    ],
    reference_frame=world.root,
)

graph_of_convex_sets = GraphOfConvexSets.free_space_from_world(
    world=world,
    search_space=search_space,
    bloat_obstacles=0.22,
)
graph_of_convex_sets.create_as_region(
    name=PrefixedName("full_kitchen_navigation_free_space")
)

start_point = navigation_point(initial_base_pose, navigation_height, world.root)
pickup_base_pose = None
pickup_point = None
raw_pickup_path = None
for candidate_base_pose in reachability_location(
    milk,
    context,
    Arms.RIGHT,
    grasp_description,
):
    candidate_point = navigation_point(
        candidate_base_pose, navigation_height, world.root
    )
    candidate_path = graph_of_convex_sets.path_from_to(
        start_point, candidate_point
    )
    if candidate_path is None:
        continue
    pickup_base_pose = candidate_base_pose
    pickup_point = candidate_point
    raw_pickup_path = candidate_path
    break

assert pickup_base_pose is not None
assert pickup_point is not None
assert raw_pickup_path is not None

sampled_placement_points = main_table.sample_points_from_surface(
    body_to_sample_for=milk_annotation,
    amount=20,
)
placement_target = None
placement_base_pose = None
placement_point = None
raw_placement_path = None
for sampled_point in sampled_placement_points:
    candidate_target = placement_pose_from_surface_sample(
        sampled_point, main_table, milk, world
    )
    for candidate_base_pose in reachability_location(
        candidate_target,
        context,
        Arms.RIGHT,
        grasp_description,
    ):
        candidate_point = navigation_point(
            candidate_base_pose, navigation_height, world.root
        )
        candidate_path = graph_of_convex_sets.path_from_to(
            pickup_point, candidate_point
        )
        if candidate_path is None:
            continue
        placement_target = candidate_target
        placement_base_pose = candidate_base_pose
        placement_point = candidate_point
        raw_placement_path = candidate_path
        break
    if placement_target is not None:
        break

assert placement_target is not None
assert placement_base_pose is not None
assert placement_point is not None
assert raw_placement_path is not None

pickup_path = filter_path(raw_pickup_path, minimum_waypoint_distance=0.08)
placement_path = filter_path(raw_placement_path, minimum_waypoint_distance=0.08)

{
    "graph_nodes": len(graph_of_convex_sets.graph.nodes()),
    "graph_edges": len(graph_of_convex_sets.graph.edges()),
    "initial_base": [round(float(value), 3) for value in initial_base_pose.to_position().to_list()[:3]],
    "pickup_base": [round(float(value), 3) for value in pickup_base_pose.to_position().to_list()[:3]],
    "sampled_placement": [round(float(value), 3) for value in placement_target.to_position().to_list()[:3]],
    "placement_base": [round(float(value), 3) for value in placement_base_pose.to_position().to_list()[:3]],
}

INFO:graph_of_convex_sets.py::543 free_space_from_semantic_annotation Free space calculated in 22.161837 ms
INFO:graph_of_convex_sets.py::561 free_space_from_semantic_annotation Connectivity calculated in 28.467412 ms


{'graph_nodes': 111,
 'graph_edges': 110,
 'initial_base': [1.5, 2.5, 0.0],
 'pickup_base': [1.75, 1.86, 0.0],
 'sampled_placement': [4.965, 3.339, 0.82],
 'placement_base': [4.885, 2.619, 0.0]}

In [8]:
figure = go.Figure(graph_of_convex_sets.plot_free_space())
figure.add_trace(
    go.Scatter3d(
        x=[point.x for point in pickup_path],
        y=[point.y for point in pickup_path],
        z=[navigation_height for point in pickup_path],
        mode="lines+markers",
        name="Pickup route",
        line={"color": "black", "width": 8},
    )
)
figure.add_trace(
    go.Scatter3d(
        x=[point.x for point in placement_path],
        y=[point.y for point in placement_path],
        z=[navigation_height + 0.01 for point in placement_path],
        mode="lines+markers",
        name="Placement route",
        line={"color": "orange", "width": 8},
    )
)
figure.add_trace(
    go.Scatter3d(
        x=[placement_target.x],
        y=[placement_target.y],
        z=[placement_target.z],
        mode="markers",
        name="Sampled table placement",
        marker={"color": "green", "size": 8},
    )
)
figure.update_layout(height=600, scene_aspectmode="data")
figure

In [9]:
navigation_step_distance = 0.08
seconds_per_navigation_step = 0.08
pickup_yaw = float(pickup_base_pose.to_quaternion().to_rpy()[2])
placement_yaw = float(placement_base_pose.to_quaternion().to_rpy()[2])

pickup_navigation_poses = interpolate_path(
    pickup_path,
    navigation_step_distance,
    pickup_yaw,
    world.root,
)
placement_navigation_poses = interpolate_path(
    placement_path,
    navigation_step_distance,
    placement_yaw,
    world.root,
)

pickup_navigation_steps = []
for navigation_pose in pickup_navigation_poses:
    pickup_navigation_steps.extend(
        [
            NavigateAction(navigation_pose),
            code(partial(time.sleep, seconds_per_navigation_step)),
        ]
    )

placement_navigation_steps = []
for navigation_pose in placement_navigation_poses:
    placement_navigation_steps.extend(
        [
            NavigateAction(navigation_pose),
            code(partial(time.sleep, seconds_per_navigation_step)),
        ]
    )

len(pickup_navigation_poses), len(placement_navigation_poses)

(9, 66)

In [10]:
task = sequential(
    [
        *pickup_navigation_steps,
        PickUpAction(milk, Arms.RIGHT, grasp_description),
        ParkArmsAction(Arms.BOTH),
        *placement_navigation_steps,
        PlaceAction(milk, placement_target, Arms.RIGHT),
        ParkArmsAction(Arms.BOTH),
    ],
    context=context,
)

with simulated_robot:
    task.perform()

In [11]:
placed_position = milk.global_pose.to_position()
placement_error = math.dist(
    [float(placed_position.x), float(placed_position.y), float(placed_position.z)],
    [float(placement_target.x), float(placement_target.y), float(placement_target.z)],
)
placed = placement_error <= 0.03
assert placed
assert milk.parent_connection.parent == world.root

{
    "graph_nodes": len(graph_of_convex_sets.graph.nodes()),
    "graph_edges": len(graph_of_convex_sets.graph.edges()),
    "pickup_navigation_updates": len(pickup_navigation_poses),
    "placement_navigation_updates": len(placement_navigation_poses),
    "placement_error_meters": round(placement_error, 4),
    "placed_at_sampled_table_pose": placed,
    "target_surface": str(main_table.supporting_surface.name),
    "cereal_remains_on_counter": cereal.parent_connection.parent == world.root,
}

{'graph_nodes': 111,
 'graph_edges': 110,
 'pickup_navigation_updates': 9,
 'placement_navigation_updates': 66,
 'placement_error_meters': 0.0052,
 'placed_at_sampled_table_pose': True,
 'target_surface': 'apartment/table_area_main_supporting_surface_region',
 'cereal_remains_on_counter': True}